# 🎬 CineMatch — EDA & Model Walkthrough
This notebook explains how each part of the system works with real examples.
Run cells top-to-bottom after placing MovieLens CSVs in `../data/`

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#131929'
matplotlib.rcParams['axes.facecolor']   = '#1c2540'
matplotlib.rcParams['text.color']       = 'white'
matplotlib.rcParams['axes.labelcolor']  = 'white'
matplotlib.rcParams['xtick.color']      = 'white'
matplotlib.rcParams['ytick.color']      = 'white'

from config import DATA_DIR
from backend.utils.preprocessing import run_pipeline, ALL_GENRES
print('✅ Imports OK')

## 1. Load & Explore Data

In [ ]:
movies_df, ratings_df = run_pipeline(DATA_DIR)
movies_df.head()

In [ ]:
# Genre distribution
from collections import Counter
genre_counts = Counter()
for genres in movies_df['genres_list']:
    genre_counts.update(genres)

labels, values = zip(*genre_counts.most_common(15))
plt.figure(figsize=(12,5))
bars = plt.bar(labels, values, color='#f5c518')
plt.title('Top Genres in MovieLens', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Ratings distribution
plt.figure(figsize=(10,4))
plt.hist(ratings_df['rating'], bins=10, color='#f5c518', edgecolor='#0a0e1a')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(ratings_df['rating'].describe())

## 2. Content-Based Filtering Demo

In [ ]:
from backend.models.content_based import ContentBasedRecommender
from config import ARTIFACTS_DIR

cb = ContentBasedRecommender(str(ARTIFACTS_DIR))

# Try loading saved model, otherwise train fresh
if not cb.load():
    cb.fit(movies_df)
    cb.save()
print(f'Vocabulary size: {len(cb.vectorizer.vocabulary_):,}')

In [ ]:
# Find movies similar to Toy Story (id=1 in MovieLens)
toy_story_id = movies_df[movies_df['title'].str.contains('Toy Story', na=False)]['id'].iloc[0]
print(f'Toy Story ID: {toy_story_id}')

similar = cb.recommend_by_movie(toy_story_id, top_n=10)
movie_lookup = movies_df.set_index('id')

print('\nMovies similar to Toy Story:')
for i, rec in enumerate(similar, 1):
    mid   = rec['movie_id']
    title = movie_lookup.loc[mid, 'title'] if mid in movie_lookup.index else '?'
    genres= movie_lookup.loc[mid, 'genres'] if mid in movie_lookup.index else '?'
    print(f'  {i:2}. {title:<40} | {genres:<40} | score={rec["score"]:.4f}')

## 3. Visualise TF-IDF Similarity (Toy Story vs others)

In [ ]:
scores = [r['score'] for r in similar]
titles = [movie_lookup.loc[r['movie_id'],'title'][:25] if r['movie_id'] in movie_lookup.index else '?' for r in similar]

plt.figure(figsize=(10,5))
plt.barh(titles[::-1], scores[::-1], color='#f5c518')
plt.xlabel('Cosine Similarity')
plt.title('Toy Story — Top 10 Similar Movies')
plt.xlim(0, max(scores)*1.1)
plt.tight_layout()
plt.show()

## 4. User Profile Demo

In [ ]:
from backend.models.user_profile import UserProfileBuilder

pb = UserProfileBuilder(cb, movies_df)

# Simulate a user who watched action movies
action_movies = movies_df[movies_df['genres'].str.contains('Action', na=False)].head(5)['id'].tolist()
genre_vec = pb.build_genre_vector(action_movies)
top5 = pb.top_genres(genre_vec)

print('Watch list (action fan):', [movie_lookup.loc[m,'title'] for m in action_movies])
print('\nTop inferred genres:', top5)

# Bar chart of genre profile
plt.figure(figsize=(12,4))
plt.bar(ALL_GENRES, genre_vec, color='#f5c518')
plt.title('User Genre Profile (action fan)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Evaluation — Precision@10, NDCG@10

In [ ]:
from backend.utils.evaluation import evaluate_recommender

def recommend_fn(user_id, watched_ids):
    if not watched_ids: return []
    return cb.recommend_by_movie(watched_ids[-1], top_n=10, exclude_ids=watched_ids)

metrics = evaluate_recommender(
    recommend_fn=recommend_fn,
    ratings_df=ratings_df,
    k=10,
    n_users=100,
)

print('\n📊 Evaluation Results:')
for k, v in metrics.items():
    print(f'   {k}: {v}')